# Extraction comparison: neuropil and background corrections for one run

The QC notebook's ΔF/F0 subtracts 0.7 of each ROI's neuropil (Suite2p's default), and in the good session about half the ROIs then have F0 ≤ 0 somewhere. That default assumes the neuropil ring carries calcium signal about as bright as the contamination inside the ROI. With **ribosome-tethered GCaMP** the indicator stays in the somata, so the neuropil holds little of it: its signal is mostly out-of-focus somata, autofluorescence and any detector offset, and subtracting 0.7 of it can remove most of a dim cell's own baseline.

This notebook loads one run that has already been registered and extracted with the QC notebook (`F.npy`, `Fneu.npy`, `stat.npy` from the Cellpose masks), and compares, on the same ROIs:

| Correction | Corrected F |
|---|---|
| Neuropil 0.7 | `F − 0.7·Fneu` (`[fluorescence] neuropil_coefficient`; the QC notebook's standard) |
| Neuropil Fitted | `F − cᵢ·Fneu`, each ROI's own `cᵢ` (see below) |
| No Neuropil | raw `F` |
| No Neuropil, Bg Regressed | raw `F` with the background (vessel mask, or every pixel outside the ROIs) regressed out per ROI, mean kept |
| Neuropil 0.7, Bg Regressed | background regressed out of `F` and `Fneu`, then `F − 0.7·Fneu` (the QC notebook's background version) |

Neuropil Fitted is each ROI's least-squares slope on its neuropil after high-passing both (60 s running mean removed, so bleaching and drift do not count), with the non-ROI background as a second regressor (so a field-wide change is not taken as neuropil; the large non-ROI region gives a cleaner trace than a small vessel mask, and a noisy trace would bias the coefficient high), without its transients. It is not a measurement of the contamination: noise in the neuropil biases it low, activity seen in both the ROI and its ring biases it high.

ΔF/F0 then uses the same baseline for all (`[fluorescence] baseline`, `win_baseline_s`, ...). Before that, two diagnostics:

1. **ROI contrast:** median F over median neuropil per ROI. Near 1 means the cells barely stand out from their surroundings (expression, indicator brightness), whatever the correction.
2. **Green vs red background:** whether the field-wide fast fluctuation is in both channels by the same fraction (laser, shared detection, focus or motion), green more than red (blood volume), or green only (indicator signal, e.g. out-of-focus somata).

It uses the `[qc]` section of the config (the run, `cellpose_seg`, `background_seg`, `valid_frames`). It does not change the extraction (`F.npy`, `Fneu.npy`, ...). It writes the background traces' caches (`background_vessel_traces.npz`, `background_non_roi_traces.npz`, shared with the QC notebook) in the plane folder, and with `SAVE_FIGURES = True` each figure as png, pdf and svg with a `.caption.txt` in `<plane>/extraction_comparison/`.

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mini2p_preprocessing import plots
from mini2p_preprocessing.background_io import SOURCE_LABELS, load_non_roi_traces, load_vessel_traces
from mini2p_preprocessing.config import (
    distortion_corrected,
    find_registered_plane,
    load_qc_run,
    load_settings,
    variant_output_dir,
)
from mini2p_preprocessing.extraction_comparison import (
    background_channel_check,
    contrast_ratio,
    corrected_traces,
    fitted_neuropil_coefficients,
    high_pass,
    method_metrics,
    valid_everywhere,
)
from mini2p_preprocessing.fluorescence import dff_from_corrected, select_frames
from mini2p_preprocessing.pipeline import QC_FONTS
from mini2p_preprocessing.plane import load_plane_traces, load_valid_frames
from mini2p_preprocessing.registration import load_registered_outputs

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s", force=True)

CONFIG = Path("../config/pipeline.toml")
SAVE_FIGURES = True  # png, pdf and svg with a caption, in <plane>/extraction_comparison/

settings = load_settings(CONFIG)
run = load_qc_run(CONFIG)
fl = settings.fluorescence
OUTPUT_DIR = variant_output_dir(
    run.run_dir, settings.destripe.enabled, distortion_corrected=distortion_corrected(settings)
)
PLANE_DIR = find_registered_plane(OUTPUT_DIR, settings.registration.motion_alignment_source,
                                  settings.registration.save_folder)
if PLANE_DIR is None:
    raise FileNotFoundError(f"No registration in {OUTPUT_DIR}; register it in the QC notebook")
FIGURES = PLANE_DIR / "extraction_comparison"

traces = load_plane_traces(PLANE_DIR)
frames = load_valid_frames(run.valid_frames, run.valid_frames_key) if run.valid_frames else None
f, fneu = select_frames(traces.f, frames), select_frames(traces.fneu, frames)
print("Plane:", PLANE_DIR)
print(f"{f.shape[0]} ROIs x {f.shape[1]} frames "
      f"({'valid frames only' if frames is not None else 'all frames'}) at {traces.fs:g} Hz")

registered = load_registered_outputs(PLANE_DIR)
backgrounds = {}  # label -> BackgroundTraces (cached in the plane folder after the first run)
if run.background_seg is None:
    print("No [qc] background_seg: the vessel background is skipped")
else:
    backgrounds["Vessel"] = load_vessel_traces(registered, PLANE_DIR, run.background_seg, settings,
                                               image=run.background_image, exclude=traces.labels > 0)
backgrounds["Non-ROI"] = load_non_roi_traces(registered, PLANE_DIR, traces.labels, settings)
for label, background in backgrounds.items():
    print(f"{label} background: {int(background.mask.sum())} pixels")


def save(fig, name, caption):
    """The figure as png, pdf and svg, and its caption, in FIGURES (if SAVE_FIGURES)."""
    if not SAVE_FIGURES:
        return
    FIGURES.mkdir(exist_ok=True)
    for extension in ("png", "pdf", "svg"):
        fig.savefig(FIGURES / f"{name}.{extension}", dpi=200, bbox_inches="tight")
    (FIGURES / f"{name}.caption.txt").write_text(caption.strip() + "\n")

## 1. How far the ROIs stand above their surroundings

Median raw F over median neuropil, per ROI. With soma-targeted GCaMP a healthy cell should be clearly brighter than its ring (ratios well above 1); ratios near 1 mean the cells barely stand out, which no correction can fix. The raw levels are printed too: if the neuropil's level is near zero or negative, the detector offset is part of the problem (a NaN ratio means a neuropil median ≤ 0).

In [ ]:
ratio = contrast_ratio(f, fneu)
f_med, fneu_med = np.median(f, axis=1), np.median(fneu, axis=1)
print(f"Each ROI's median F: median over ROIs {np.median(f_med):.1f} "
      f"(5th-95th percentile {np.percentile(f_med, 5):.1f}-{np.percentile(f_med, 95):.1f})")
print(f"Each ROI's median neuropil: median over ROIs {np.median(fneu_med):.1f} "
      f"(5th-95th percentile {np.percentile(fneu_med, 5):.1f}-{np.percentile(fneu_med, 95):.1f})")
print(f"Lowest raw F anywhere: {f.min():.1f}; lowest neuropil: {fneu.min():.1f}")
print(f"ROIs with no positive neuropil median: {int(np.isnan(ratio).sum())}")
for cut in (1.1, 1.25, 1.5):
    print(f"ROIs with contrast below {cut}: {int((ratio < cut).sum())} of {len(ratio)}")
with plt.rc_context({"font.family": "sans-serif", "font.sans-serif": QC_FONTS}):
    ax = plots.plot_contrast_ratio(ratio)
    save(ax.figure, "roi_contrast", """
Median raw F over median neuropil (Fneu) for each ROI, over the frames analysed. Dashed: 1,
a ROI no brighter than its surrounding neuropil ring. ROIs whose neuropil median is not
positive have no ratio and are left out (n counts the rest).""")
    plt.show()

## 2. The field-wide fluctuation: green and red

Each background trace, high-passed (its 60 s running mean removed, so bleaching and slow drift, which differ between channels, do not count), as a fraction of its mean above the detector's dark level (`DARK_GREEN`, `DARK_RED`: the value a pixel reads with no light, e.g. with the laser blanked; 0 if not known, which compresses the fractions of a channel with an offset). Then green's fractional change regressed on red's:

- **slope near 1**: the part of green's fluctuation that red shares moves both channels by the same fraction: laser power, shared detection, or **focus and motion**, which in a freely moving mouse can do this too;
- **slope above 1**: green moves more than red for the shared part, as with **blood volume** (hemoglobin absorbs green more than red);
- **r²**: the fraction of green's fluctuation shared with red; the rest is in green alone, e.g. **indicator signal of out-of-focus somata** (or green's noise).

Signal in green alone does not bias the slope (it only lowers r²), but noise in red pulls it towards 0, and the red channel was unreliable in some sessions. So the slope is given as a range, from green-on-red to the reverse regression (slope / r²): the shared part's true slope lies between the two. Read "near 1" only when the whole range is near 1; a wide range (low r²) cannot tell hardware from blood volume.

In [ ]:
DARK_GREEN, DARK_RED = 0.0, 0.0  # detector dark level of each channel, if known

channel_checks = {}
for label, background in backgrounds.items():
    green_bg, red_bg = select_frames(background.green, frames), select_frames(background.red, frames)
    check = background_channel_check(green_bg, red_bg, traces.fs, DARK_GREEN, DARK_RED)
    channel_checks[label] = check
    print(f"{label}: green-on-red slope {check['slope_low']:.2f} to {check['slope_high']:.2f}, "
          f"r² = {check['r_squared']:.2f}, "
          f"SD green {100 * check['green_sd_fraction']:.1f}% / red {100 * check['red_sd_fraction']:.1f}% of the mean")
    with plt.rc_context({"font.family": "sans-serif", "font.sans-serif": QC_FONTS}):
        axes = plots.plot_background_channels(green_bg, red_bg, traces.fs, check, DARK_GREEN, DARK_RED)
        axes[0].figure.suptitle(f"{label} Background")
        save(axes[0].figure, f"background_green_red_{label.lower().replace('-', '_')}", f"""
{label} background (mean of its pixels per frame), green and red, each high-passed (60 s
running mean removed) and as a fraction of its mean above the dark level (green {DARK_GREEN:g},
red {DARK_RED:g}). Left: over time. Right: green against red (one point per frame, subsampled)
with the least-squares slope of green on red through zero (solid) and slope 1 (dashed). The
title gives the slope's range, from green-on-red to the reverse regression (slope / r²); the
shared part's true slope lies between them. Near 1 throughout: the shared part changes both
channels by the same fraction (laser, detection, focus or motion); above 1: green changes more
(e.g. blood volume). r² = {check['r_squared']:.2f} is the fraction of green's fluctuation shared
with red; a low r² gives a wide range that cannot tell these apart.""")
        plt.show()
if len(background_green_hp := {label: high_pass(select_frames(b.green, frames), traces.fs)
                                for label, b in backgrounds.items()}) == 2:
    r_backgrounds = np.corrcoef(*background_green_hp.values())[0, 1]
    print(f"Vessel vs non-ROI background (green, high-passed): r = {r_backgrounds:.3f}")

## 3. The corrections

`FIT_TRANSIENT_SD`: frames more than this many robust SDs above the fit are left out of each ROI's later neuropil fits (its transients). The fit uses the non-ROI background as its second regressor: from many more pixels than the vessel mask, its trace is less noisy, and a noisy one would bias the coefficients high. ΔF/F0 is undefined where F0 ≤ `[fluorescence] min_f0`. Noise and skew are compared on the ROIs whose F0 stays valid under every correction (`common`), so every correction is summarised over the same cells; with no such ROI they are left blank and only the baseline counts are given. Computing every baseline takes a minute or two.

In [ ]:
FIT_TRANSIENT_SD = 2.0

background_green = {label: select_frames(b.green, frames) for label, b in backgrounds.items()}
reference = background_green.get("Vessel", background_green["Non-ROI"])
fitted = fitted_neuropil_coefficients(f, fneu, traces.fs, background=background_green["Non-ROI"],
                                      transient_sd=FIT_TRANSIENT_SD)
print(f"Fitted neuropil coefficients: median {np.median(fitted):.2f} "
      f"[{np.percentile(fitted, 5):.2f}, {np.percentile(fitted, 95):.2f}] (5th-95th percentile); "
      f"{int((fitted == 0).sum())} at 0, {int((fitted == 1).sum())} at 1")
with plt.rc_context({"font.family": "sans-serif", "font.sans-serif": QC_FONTS}):
    ax = plots.plot_fitted_coefficients(fitted, fl.neuropil_coefficient)
    save(ax.figure, "fitted_neuropil_coefficients", f"""
Each ROI's fitted neuropil coefficient: the least-squares slope of its F on its neuropil, both
high-passed (60 s running mean removed), with the non-ROI background as a second regressor; the later
fits leave out frames more than {FIT_TRANSIENT_SD:g} robust SDs above the previous fit (the
cell's transients); kept in [0, 1]. Neuropil noise biases it low, activity seen in both the ROI
and its ring biases it high. Dashed: the fixed coefficient, {fl.neuropil_coefficient:g}.""")
    plt.show()

corrected = corrected_traces(f, fneu, background_green, fl.neuropil_coefficient, fitted)
results = {name: dff_from_corrected(trace, traces.fs, fl) for name, trace in corrected.items()}
common = valid_everywhere(results)
print(f"{int(common.sum())} of {len(common)} ROIs have a valid F0 under every correction")
if not common.any():
    print("No common ROI: noise and skew are left blank; compare the baseline counts")
metrics = {name: method_metrics(result, reference, traces.fs, common)
           for name, result in results.items()}

keys = list(next(iter(metrics.values())))
SHORT = {"rois_invalid_f0_anywhere": "bad F0 anywhere", "rois_invalid_f0_mostly": "bad F0 mostly",
         "fraction_frames_invalid_f0": "frames no ΔF/F0", "median_f0": "median F0",
         "median_noise": "noise", "median_skew": "skew", "median_abs_r_background": "|r| bg",
         "mean_pairwise_r": "pairwise r"}
width = max(len(name) for name in metrics)
print(f"\n{'':{width}}  " + "  ".join(f"{SHORT[k]:>15}" for k in keys))
for name, row in metrics.items():
    print(f"{name:{width}}  " + "  ".join(f"{row[k]:>15.3g}" for k in keys))
print(f"\nbad F0: F0 <= {fl.min_f0:g}; noise and skew over the {int(common.sum())} common ROIs; "
      f"|r| with the {'vessel' if 'Vessel' in background_green else 'non-ROI'} background, high-passed")

**How to read it.** A correction that suits soma-targeted GCaMP should leave few ROIs with a bad F0 and keep positive skew (transients) on the common ROIs.

- **Noise** is in ΔF/F0 units, so it is not comparable on its own: without neuropil subtraction F0 includes the surrounding light, which shrinks both noise and transients in ΔF/F0. Skew is less affected but not immune: a small but positive F0 gives large ΔF/F0 values.
- **|r| with the background** is near zero by construction for the corrections that regress out a background (and nearly so for the other background when the two are as correlated as printed in section 2); use it to compare the others, i.e. how much fast field-wide signal each leaves in.
- **Pairwise r**: removing shared signal also removes real population activity, so a low pairwise r is not the goal on its own.

If every correction leaves many ROIs with a bad F0, or the ROI contrast above is near 1, the cells are too dim relative to their surroundings, and the fix is expression or brightness, not analysis.

In [ ]:
with plt.rc_context({"font.family": "sans-serif", "font.sans-serif": QC_FONTS}):
    ax = plots.plot_f0_by_method({name: np.median(r.f0, axis=1) for name, r in results.items()},
                                fl.min_f0)
    save(ax.figure, "f0_by_correction", f"""
Cumulative distribution over ROIs of each ROI's median baseline F0, for each correction (the
same ROIs and frames, the same baseline method and settings). Dashed: min_f0
({fl.min_f0:g}), at and below which ΔF/F0 is undefined.""")
    plt.show()
    axes = plots.plot_method_metrics(metrics, keys, fl.min_f0)
    axes[0].figure.suptitle("Corrections Compared")
    save(axes[0].figure, "corrections_compared", f"""
For each correction: ROIs whose F0 is <= min_f0 ({fl.min_f0:g}) in any frame and in at least
half of the frames, and the fraction of ROI-frames without a ΔF/F0 (all ROIs); the median over
ROIs of each ROI's median F0; the median frame-to-frame SD of ΔF/F0 (robust SD of frame
differences / sqrt 2) and median skewness of ΔF/F0, over the {int(common.sum())} ROIs whose F0
is valid under every correction; the median |r| between each ROI's corrected F and the
background (vessel if there is one, else non-ROI), both high-passed (near zero by construction
for corrections that regress out a background); the mean correlation between ROIs' high-passed
corrected F (up to 300 ROIs, evenly spaced).""")
    plt.show()

## 4. The same ROIs under each correction

The `N_SHOWN` ROIs with the largest 99th percentile of ΔF/F0 under `RANK_BY`, among those whose F0 stays valid there, drawn the same way under every correction on one vertical scale. A ROI whose F0 is ≤ `min_f0` under a correction has gaps there.

In [ ]:
N_SHOWN = 12
RANK_BY = next((name for name in results if name.startswith("No Neuropil, ")), "Neuropil Fitted")
SHOWN_METHODS = list(results)  # or a few names from the table above

ranked = results[RANK_BY]
valid = ~ranked.invalid_f0.any(axis=1)
p99 = np.full(len(valid), -np.inf)
p99[valid] = np.nanpercentile(ranked.dff[valid], 99, axis=1)
if not valid.any():
    raise ValueError(f"No ROI has a valid F0 throughout under {RANK_BY}; choose another RANK_BY")
rois = np.argsort(-p99, kind="stable")[: min(N_SHOWN, int(valid.sum()))]
print(f"Ranked by {RANK_BY}: ROIs {rois.tolist()}")
with plt.rc_context({"font.family": "sans-serif", "font.sans-serif": QC_FONTS}):
    _, panels = plt.subplots(1, len(SHOWN_METHODS), figsize=(min(3.5 * len(SHOWN_METHODS), 24), 9),
                             constrained_layout=True, squeeze=False)
    axes = plots.plot_top_rois({name: results[name].dff for name in SHOWN_METHODS}, rois,
                               axes=panels, xlabel="Valid Frames" if frames is not None else "Frames")
    axes[0].figure.suptitle(f"Top {len(rois)} ROIs By 99th Percentile ΔF/F0 ({RANK_BY})")
    save(axes[0].figure, "top_rois_by_correction", f"""
The {len(rois)} ROIs with the largest 99th percentile of ΔF/F0 under "{RANK_BY}", among
ROIs whose F0 stays above min_f0 under it, shown under each correction (one panel each) on one
vertical scale (bar: one trace spacing, in ΔF/F0). Gaps: frames where that correction's F0
is <= min_f0 ({fl.min_f0:g}).""")
    plt.show()